# Phishing Message Detection using NLP
**Atul Biju — S7 AI — Roll number 21**

This notebook reproduces the leakage-aware raw-text versus NLTK comparison. Run it from the project root after installing `requirements.txt` and downloading the pinned dataset.

In [ ]:
from pathlib import Path
import json, sys
sys.path.insert(0, str(Path.cwd() / 'src'))
from phishdetect.data import load_jsonl, stratified_split
from phishdetect.modeling import build_pipeline, metrics
from phishdetect.inference import predict_message

## Data safeguards
The loader preserves the publisher's `phishing`/`benign` labels, joins subject and body, removes exact text duplicates, then creates a fixed-seed stratified 60/20/20 split. TF-IDF remains inside each pipeline, so it sees training data only during fitting.

In [ ]:
frame = load_jsonl('data/raw/phishing_benign.jsonl')
train, validation, test = stratified_split(frame, seed=42)
{'rows': len(frame), 'duplicates_removed': frame.attrs['duplicates_removed'], 'class_counts': frame.label.value_counts().to_dict(), 'splits': [len(train), len(validation), len(test)]}

## Raw text versus NLTK preprocessing
The NLTK path uses Treebank tokenization, WordNet lemmatization, URL/email placeholders, and selective stopword filtering that retains negation.

In [ ]:
models = {'raw_text': build_pipeline(False), 'nltk_preprocessed': build_pipeline(True)}
results = {}
for name, model in models.items():
    model.fit(train.text, train.target)
    results[name] = {'validation': metrics(model, validation), 'test': metrics(model, test)}
results

In [ ]:
selected_name = max(models, key=lambda name: (results[name]['validation']['f1'], results[name]['validation']['recall']))
selected = models[selected_name]
selected_name, predict_message(selected, 'Urgent: verify your password at https://example.invalid now')

## Interpretation
The recorded run selected the NLTK pipeline and scored 1.0000 F1 on a 40-message test set. This is an optimistic result on a tiny, balanced, synthetic/curated corpus with strong templates—not production evidence. The displayed probability is uncalibrated, and coefficient contributions are model-derived associations rather than causal explanations.